In [1]:
from IPython.display import HTML
from sympy import symbols, pretty
from sympy.logic.boolalg import truth_table

def afficher_table_verite(expressions, variables, noms=None):
    """
    Affiche la table de vérité pour une ou plusieurs expressions logiques.
    
    Paramètres:
        expressions : une expression SymPy OU une liste d'expressions
        variables   : liste des variables (ex: [p, q])
        noms        : nom (str) ou liste de noms pour les colonnes d'expressions.
                      Si None, utilise pretty(expr) pour chaque expression.
    """
    # Normaliser en liste
    if not isinstance(expressions, (list, tuple)):
        expressions = [expressions]
    
    if noms is None:
        noms = [pretty(e) for e in expressions]
    elif isinstance(noms, str):
        noms = [noms]
    
    if len(noms) != len(expressions):
        raise ValueError(f"Nombre de noms ({len(noms)}) != nombre d'expressions ({len(expressions)})")
    
    style_th = "background:#3498db;color:white;padding:6px 12px;border:1px solid #2980b9"
    style_td = "padding:6px 12px;border:1px solid #ddd;text-align:center"
    
    # En-têtes
    html = "<table style='border-collapse:collapse;font-family:sans-serif'>"
    html += "<tr>"
    for v in variables:
        html += f"<th style='{style_th}'>{v}</th>"
    for nom in noms:
        html += f"<th style='{style_th}'>{nom}</th>"
    html += "</tr>"
    
    # Calculer toutes les tables de vérité en parallèle
    # truth_table renvoie un générateur, on récupère les résultats par expression
    resultats = [list(truth_table(e, variables)) for e in expressions]
    
    # Toutes les expressions partagent les mêmes vL (même ordre des variables)
    nb_lignes = len(resultats[0])
    
    for i in range(nb_lignes):
        vL = resultats[0][i][0]  # valeurs des variables (identiques pour toutes les exprs)
        html += "<tr>"
        for val in vL:
            html += f"<td style='{style_td}'>{val}</td>"
        for j in range(len(expressions)):
            vr = resultats[j][i][1]
            couleur = "#27ae60" if vr else "#c0392b"
            html += f"<td style='{style_td};color:{couleur};font-weight:bold'>{vr}</td>"
        html += "</tr>"
    html += "</table>"
    return HTML(html)

In [2]:
p, q = symbols('p q')

# Une seule expression (compatibilité avec l'ancienne signature)
afficher_table_verite(p >> q, [p, q], 'p → q')

p,q,p → q
0,0,True
0,1,True
1,0,False
1,1,True


In [3]:

# Plusieurs expressions, noms automatiques
afficher_table_verite([p & q, p | q, p >> q, ~p], [p, q])


p,q,p ∧ q,p ∨ q,p → q,¬p
0,0,False,False,True,True
0,1,False,True,True,True
1,0,False,True,False,False
1,1,True,True,True,False


In [4]:

# Plusieurs expressions avec noms personnalisés
afficher_table_verite(
    [p & q, p | q, p >> q, (p >> q) & (q >> p)],
    [p, q],
    ['p ∧ q', 'p ∨ q', 'p → q', 'p ↔ q']
)


p,q,p ∧ q,p ∨ q,p → q,p ↔ q
0,0,False,False,True,True
0,1,False,True,True,False
1,0,False,True,False,False
1,1,True,True,True,True


In [5]:

# Utile pour vérifier des équivalences logiques (lois de De Morgan)
p, q = symbols('p q')
afficher_table_verite(
    [~(p & q), ~p | ~q, ~(p | q), ~p & ~q],
    [p, q],
    ['¬(p ∧ q)', '¬p ∨ ¬q', '¬(p ∨ q)', '¬p ∧ ¬q']
)


p,q,¬(p ∧ q),¬p ∨ ¬q,¬(p ∨ q),¬p ∧ ¬q
0,0,True,True,True,True
0,1,True,True,False,False
1,0,True,True,False,False
1,1,False,False,False,False


In [6]:

# Avec 3 variables
p, q, r = symbols('p q r')
afficher_table_verite(
    [(p & q) | r, p & (q | r)],
    [p, q, r],
    ['(p ∧ q) ∨ r', 'p ∧ (q ∨ r)']
)

p,q,r,(p ∧ q) ∨ r,p ∧ (q ∨ r)
0,0,0,False,False
0,0,1,True,False
0,1,0,False,False
0,1,1,True,False
1,0,0,False,False
1,0,1,True,True
1,1,0,True,True
1,1,1,True,True


In [7]:
from sympy import symbols, simplify_logic, Equivalent
from sympy.logic.boolalg import truth_table

def equivalent(expr1, expr2):
    """Vérifie si deux expressions logiques sont équivalentes."""
    return simplify_logic(Equivalent(expr1, expr2)) == True

p, q = symbols('p q')
print(equivalent(p >> q, ~p | q))           # True
print(equivalent(~(p & q), ~p | ~q))        # True (De Morgan)
print(equivalent(p & q, p | q))             # False

True
True
False


In [8]:
from IPython.display import HTML
from sympy import symbols, simplify_logic, Equivalent, pretty, latex

class DemonstrationLogique:
    """
    Permet d'écrire une démonstration par équivalences successives.
    Chaque étape est vérifiée automatiquement.
    """
    def __init__(self, expr_initiale, titre="Démonstration"):
        self.titre = titre
        self.etapes = [(expr_initiale, "Expression initiale")]
        self.erreurs = []
    
    def egal(self, nouvelle_expr, justification):
        """Ajoute une étape : nouvelle_expr doit être équivalente à la précédente."""
        precedente = self.etapes[-1][0]
        if simplify_logic(Equivalent(precedente, nouvelle_expr)) == True:
            self.etapes.append((nouvelle_expr, justification))
        else:
            self.erreurs.append(
                f"❌ Étape {len(self.etapes)} INVALIDE : "
                f"{precedente}  ≢  {nouvelle_expr}  ({justification})"
            )
            self.etapes.append((nouvelle_expr, f"⚠️ {justification} [ERREUR]"))
        return self
    
    def conclure(self, expr_attendue=None):
        """Affiche la démonstration et vérifie le résultat final si fourni."""
        finale = self.etapes[-1][0]
        ok_final = True
        if expr_attendue is not None:
            ok_final = simplify_logic(Equivalent(finale, expr_attendue)) == True
        
        couleur_bord = "#27ae60" if (not self.erreurs and ok_final) else "#c0392b"
        html = f"""
        <div style='border-left:4px solid {couleur_bord};padding:12px 16px;
                    background:#f8f9fa;font-family:sans-serif;margin:8px 0'>
        <div style='font-weight:bold;color:#2c3e50;margin-bottom:8px'>{self.titre}</div>
        <table style='border-collapse:collapse'>
        """
        for i, (expr, justif) in enumerate(self.etapes):
            symbole = "" if i == 0 else "≡"
            erreur = "[ERREUR]" in justif
            couleur = "#c0392b" if erreur else "#2c3e50"
            html += f"""
            <tr>
              <td style='padding:4px 12px;color:#7f8c8d;font-family:monospace'>{symbole}</td>
              <td style='padding:4px 12px;font-family:monospace;color:{couleur}'>${latex(expr)}$</td>
              <td style='padding:4px 12px;color:#7f8c8d;font-style:italic'>{justif}</td>
            </tr>
            """
        html += "</table>"
        
        if self.erreurs:
            html += "<div style='margin-top:8px;color:#c0392b'>"
            for e in self.erreurs:
                html += f"<div>{e}</div>"
            html += "</div>"
        elif expr_attendue is not None:
            if ok_final:
                html += "<div style='margin-top:8px;color:#27ae60'>✓ Démonstration correcte</div>"
            else:
                html += f"<div style='margin-top:8px;color:#c0392b'>✗ Résultat final incorrect (attendu : ${latex(expr_attendue)}$)</div>"
        
        html += "</div>"
        return HTML(html)

In [9]:
p, q = symbols('p q')

# Démontrer que ¬(p → q) ≡ p ∧ ¬q
demo = DemonstrationLogique(p >> q, "Négation de l'implication")
demo.egal(~p | q,       "définition de l'implication : p → q ≡ ¬p ∨ q")
demo.egal(~p | ~~q,        "Double Négation")
demo.egal(~~q| ~p,        "Commutativity")
demo.egal(~q>>~p,           "double négation")
demo.conclure(~q>>~p)

,$p \Rightarrow q$,Expression initiale
≡,$q \vee \neg p$,définition de l'implication : p → q ≡ ¬p ∨ q
≡,$q \vee \neg p$,Double Négation
≡,$q \vee \neg p$,Commutativity
≡,$\neg q \Rightarrow \neg p$,double négation


In [10]:
p, q = symbols('p q')

# Démontrer que ¬(p → q) ≡ p ∧ ¬q
demo = DemonstrationLogique(~(p >> q), "Négation de l'implication")
demo.conclure(p & ~q)

,$p \not\Rightarrow q$,Expression initiale


In [11]:
# Avec une erreur volontaire
demo = DemonstrationLogique(p & (q | ~p), "Test")
demo.egal((p & q) | (p & ~p), "distributivité")    # ✓ correct
demo.egal((p & q) | p,         "absorption")        # ❌ ERREUR détectée
demo.conclure()

,$p \wedge \left(q \vee \neg p\right)$,Expression initiale
≡,$\left(p \wedge q\right) \vee \left(p \wedge \neg p\right)$,distributivité
≡,$p \vee \left(p \wedge q\right)$,⚠️ absorption [ERREUR]


In [12]:
from sympy import symbols, Not, And, Or, Implies, simplify_logic, Equivalent

# Catalogue des lois de la logique propositionnelle
LOIS = {
    "implication":      lambda p, q: Equivalent(Implies(p, q), Or(Not(p), q)),
    "de_morgan_et":     lambda p, q: Equivalent(Not(And(p, q)), Or(Not(p), Not(q))),
    "de_morgan_ou":     lambda p, q: Equivalent(Not(Or(p, q)), And(Not(p), Not(q))),
    "double_negation":  lambda p:    Equivalent(Not(Not(p)), p),
    "distributivite_et": lambda p, q, r: Equivalent(And(p, Or(q, r)), Or(And(p, q), And(p, r))),
    "distributivite_ou": lambda p, q, r: Equivalent(Or(p, And(q, r)), And(Or(p, q), Or(p, r))),
    "absorption_et":    lambda p, q: Equivalent(And(p, Or(p, q)), p),
    "absorption_ou":    lambda p, q: Equivalent(Or(p, And(p, q)), p),
    "idempotence_et":   lambda p:    Equivalent(And(p, p), p),
    "idempotence_ou":   lambda p:    Equivalent(Or(p, p), p),
    "commutativite_et": lambda p, q: Equivalent(And(p, q), And(q, p)),
    "commutativite_ou": lambda p, q: Equivalent(Or(p, q), Or(q, p)),
}

def verifier_loi(nom_loi, *args):
    """Vérifie qu'une loi du catalogue est bien une tautologie."""
    if nom_loi not in LOIS:
        return f"Loi inconnue : {nom_loi}"
    return simplify_logic(LOIS[nom_loi](*args)) == True

# Test pédagogique
p, q, r = symbols('p q r')
for nom in LOIS:
    sig = LOIS[nom].__code__.co_argcount
    args = [p, q, r][:sig]
    print(f"{nom:25s} → {verifier_loi(nom, *args)}")

implication               → True
de_morgan_et              → True
de_morgan_ou              → True
double_negation           → True
distributivite_et         → True
distributivite_ou         → True
absorption_et             → True
absorption_ou             → True
idempotence_et            → True
idempotence_ou            → True
commutativite_et          → True
commutativite_ou          → True


In [38]:
# Énoncé : Démontrer que (p → q) ∧ (p → ¬q) ≡ ¬p
# en complétant les étapes ci-dessous
demo = DemonstrationLogique((p >> q) & (p >> ~q))
demo.egal((~p|q)&(~p|~q), "définition de l'implication (×2)")
demo.egal(((~p|q)&~p|(~p|q)&~q), "distributivité de ")
demo.egal(((~p&~p|q&~p)|(~p&~q|q&~q)), "distributivité de &")
demo.egal((~p&~p|q&~p)|(~p&~q|q&~q), "distributivité de &")
demo.egal((~p|q&~p)|(~p&~q|q&~q), "identity of &")
demo.egal(~p|q&~p|(~p&~q), "identity of &")
demo.egal(~p|(~p&(q|~q)), "identity of &")
demo.egal(~p|~p, "identity of &")
demo.egal(~p|~p, "idempotency")
demo.conclure(~p)

,$\left(p \Rightarrow q\right) \wedge \left(p \Rightarrow \neg q\right)$,Expression initiale
≡,$\left(q \vee \neg p\right) \wedge \left(\neg p \vee \neg q\right)$,définition de l'implication (×2)
≡,$\left(\neg p \wedge \left(q \vee \neg p\right)\right) \vee \left(\neg q \wedge \left(q \vee \neg p\right)\right)$,distributivité de
≡,$\left(q \wedge \neg p\right) \vee \left(q \wedge \neg q\right) \vee \left(\neg p \wedge \neg q\right) \vee \neg p$,distributivité de &
≡,$\left(q \wedge \neg p\right) \vee \left(q \wedge \neg q\right) \vee \left(\neg p \wedge \neg q\right) \vee \neg p$,distributivité de &
≡,$\left(q \wedge \neg p\right) \vee \left(q \wedge \neg q\right) \vee \left(\neg p \wedge \neg q\right) \vee \neg p$,identity of &
≡,$\left(q \wedge \neg p\right) \vee \left(\neg p \wedge \neg q\right) \vee \neg p$,identity of &
≡,$\left(\neg p \wedge \left(q \vee \neg q\right)\right) \vee \neg p$,identity of &
≡,$\neg p$,identity of &
≡,$\neg p$,idempotency


In [39]:
from sympy import symbols, simplify_logic, Implies, And, Or, Not
from sympy.logic.boolalg import truth_table

def inference_valide(premisses, conclusion):
    """
    Vérifie si la conclusion découle logiquement des prémisses.
    inference_valide([P1, P2, ...], C)  ⟺  (P1 ∧ P2 ∧ ...) → C est une tautologie
    """
    if not premisses:
        return simplify_logic(conclusion) == True
    conjonction = premisses[0]
    for p in premisses[1:]:
        conjonction = And(conjonction, p)
    return simplify_logic(Implies(conjonction, conclusion)) == True

# Exemples
p, q, r = symbols('p q r')

# Modus Ponens : p, p → q ⊢ q
print(inference_valide([p, p >> q], q))           # True

# Modus Tollens : p → q, ¬q ⊢ ¬p
print(inference_valide([p >> q, ~q], ~p))         # True

# Syllogisme hypothétique : p → q, q → r ⊢ p → r
print(inference_valide([p >> q, q >> r], p >> r)) # True

# Inférence INVALIDE (affirmation du conséquent) : p → q, q ⊢ p
print(inference_valide([p >> q, q], p))           # False

True
True
True
False


In [40]:
from sympy import symbols, And, Or, Not, Implies, Equivalent, simplify_logic

REGLES_INFERENCE = {
    # Format : nom → (description, fonction(args) → (premisses, conclusion))
    "modus_ponens": {
        "description": "p, p → q  ⊢  q",
        "schema": lambda p, q: ([p, Implies(p, q)], q)
    },
    "modus_tollens": {
        "description": "p → q, ¬q  ⊢  ¬p",
        "schema": lambda p, q: ([Implies(p, q), Not(q)], Not(p))
    },
    "syllogisme_hypothetique": {
        "description": "p → q, q → r  ⊢  p → r",
        "schema": lambda p, q, r: ([Implies(p, q), Implies(q, r)], Implies(p, r))
    },
    "syllogisme_disjonctif": {
        "description": "p ∨ q, ¬p  ⊢  q",
        "schema": lambda p, q: ([Or(p, q), Not(p)], q)
    },
    "addition": {
        "description": "p  ⊢  p ∨ q",
        "schema": lambda p, q: ([p], Or(p, q))
    },
    "simplification": {
        "description": "p ∧ q  ⊢  p",
        "schema": lambda p, q: ([And(p, q)], p)
    },
    "conjonction": {
        "description": "p, q  ⊢  p ∧ q",
        "schema": lambda p, q: ([p, q], And(p, q))
    },
    "resolution": {
        "description": "p ∨ q, ¬p ∨ r  ⊢  q ∨ r",
        "schema": lambda p, q, r: ([Or(p, q), Or(Not(p), r)], Or(q, r))
    },
    "dilemme_constructif": {
        "description": "(p → q) ∧ (r → s), p ∨ r  ⊢  q ∨ s",
        "schema": lambda p, q, r, s: (
            [And(Implies(p, q), Implies(r, s)), Or(p, r)],
            Or(q, s)
        )
    },
}

def verifier_regle(nom_regle, *args):
    """Vérifie qu'une règle du catalogue est bien valide."""
    if nom_regle not in REGLES_INFERENCE:
        return False, "Règle inconnue"
    premisses, conclusion = REGLES_INFERENCE[nom_regle]["schema"](*args)
    return inference_valide(premisses, conclusion), REGLES_INFERENCE[nom_regle]["description"]

# Vérification de toutes les règles
p, q, r, s = symbols('p q r s')
print("Vérification des règles d'inférence :\n")
for nom in REGLES_INFERENCE:
    nb_args = REGLES_INFERENCE[nom]["schema"].__code__.co_argcount
    args = [p, q, r, s][:nb_args]
    valide, desc = verifier_regle(nom, *args)
    print(f"  {nom:28s} : {desc:35s} → {'✓' if valide else '✗'}")

Vérification des règles d'inférence :

  modus_ponens                 : p, p → q  ⊢  q                      → ✓
  modus_tollens                : p → q, ¬q  ⊢  ¬p                    → ✓
  syllogisme_hypothetique      : p → q, q → r  ⊢  p → r              → ✓
  syllogisme_disjonctif        : p ∨ q, ¬p  ⊢  q                     → ✓
  addition                     : p  ⊢  p ∨ q                         → ✓
  simplification               : p ∧ q  ⊢  p                         → ✓
  conjonction                  : p, q  ⊢  p ∧ q                      → ✓
  resolution                   : p ∨ q, ¬p ∨ r  ⊢  q ∨ r             → ✓
  dilemme_constructif          : (p → q) ∧ (r → s), p ∨ r  ⊢  q ∨ s  → ✓


In [41]:
from IPython.display import HTML
from sympy import symbols, simplify_logic, Implies, And, latex

class Deduction:
    """
    Permet d'écrire une dérivation par règles d'inférence.
    Chaque ligne est justifiée par les numéros des lignes utilisées + la règle.
    """
    def __init__(self, premisses, conclusion_visee, titre="Déduction"):
        self.titre = titre
        self.lignes = []  # liste de (formule, justification, indices_utilises)
        self.conclusion_visee = conclusion_visee
        self.erreurs = []
        
        # Numéroter les prémisses comme lignes 1, 2, ...
        for prem in premisses:
            self.lignes.append((prem, "Prémisse", []))
    
    def deduire(self, formule, regle, depuis):
        """
        Ajoute une nouvelle ligne déduite.
        - formule  : la formule SymPy déduite
        - regle    : nom de la règle utilisée (str)
        - depuis   : liste des numéros de ligne (1-indexés) utilisés
        """
        # Récupérer les formules sources
        try:
            sources = [self.lignes[i-1][0] for i in depuis]
        except IndexError:
            self.erreurs.append(f"Ligne {len(self.lignes)+1} : référence invalide {depuis}")
            self.lignes.append((formule, f"⚠️ {regle} depuis {depuis} [LIGNE INEXISTANTE]", depuis))
            return self
        
        # Vérifier que (sources) ⊢ formule est valide
        if inference_valide(sources, formule):
            self.lignes.append((formule, f"{regle} ({', '.join(map(str, depuis))})", depuis))
        else:
            self.erreurs.append(
                f"Ligne {len(self.lignes)+1} INVALIDE : {sources} ⊬ {formule} par {regle}"
            )
            self.lignes.append((formule, f"⚠️ {regle} ({', '.join(map(str, depuis))}) [INVALIDE]", depuis))
        return self
    
    def conclure(self):
        """Affiche la dérivation et vérifie qu'on a bien atteint la conclusion visée."""
        derniere = self.lignes[-1][0]
        atteint = simplify_logic(Equivalent(derniere, self.conclusion_visee)) == True
        couleur = "#27ae60" if (not self.erreurs and atteint) else "#c0392b"
        
        html = f"""
        <div style='border-left:4px solid {couleur};padding:12px 16px;
                    background:#f8f9fa;font-family:sans-serif;margin:8px 0'>
        <div style='font-weight:bold;color:#2c3e50;margin-bottom:8px'>{self.titre}</div>
        <div style='color:#7f8c8d;margin-bottom:8px'>
          But : montrer que les prémisses entraînent <b>${latex(self.conclusion_visee)}$</b>
        </div>
        <table style='border-collapse:collapse'>
        <tr style='border-bottom:1px solid #bdc3c7;color:#7f8c8d;font-size:0.9em'>
          <th style='padding:4px 12px;text-align:right'>n°</th>
          <th style='padding:4px 12px;text-align:left'>Formule</th>
          <th style='padding:4px 12px;text-align:left'>Justification</th>
        </tr>
        """
        for i, (formule, justif, _) in enumerate(self.lignes, start=1):
            erreur = "INVALIDE" in justif or "INEXISTANTE" in justif
            c = "#c0392b" if erreur else "#2c3e50"
            html += f"""
            <tr>
              <td style='padding:4px 12px;text-align:right;color:#7f8c8d;font-family:monospace'>{i}.</td>
              <td style='padding:4px 12px;font-family:monospace;color:{c}'>${latex(formule)}$</td>
              <td style='padding:4px 12px;color:#7f8c8d;font-style:italic'>{justif}</td>
            </tr>
            """
        html += "</table>"
        
        if self.erreurs:
            html += "<div style='margin-top:8px;color:#c0392b'>"
            for e in self.erreurs:
                html += f"<div>❌ {e}</div>"
            html += "</div>"
        elif atteint:
            html += "<div style='margin-top:8px;color:#27ae60'>✓ Dérivation correcte — conclusion atteinte</div>"
        else:
            html += f"<div style='margin-top:8px;color:#c0392b'>✗ Conclusion visée non atteinte</div>"
        
        html += "</div>"
        return HTML(html)

In [42]:
p, q, r = symbols('p q r')

deriv = Deduction(
    premisses=[p >> q, q >> r, p],
    conclusion_visee=r,
    titre="Exemple : transitivité de l'implication"
)
deriv.deduire(q,           "Modus Ponens",           depuis=[3, 1])  # p, p→q ⊢ q
deriv.deduire(r,           "Modus Ponens",           depuis=[4, 2])  # q, q→r ⊢ r
deriv.conclure()

n°,Formule,Justification
1.,$p \Rightarrow q$,Prémisse
2.,$q \Rightarrow r$,Prémisse
3.,$p$,Prémisse
4.,$q$,"Modus Ponens (3, 1)"
5.,$r$,"Modus Ponens (4, 2)"


In [43]:
deriv = Deduction(
    premisses=[p | q, ~p, q >> r],
    conclusion_visee=r,
    titre="Exercice : déduction avec disjonction"
)
deriv.deduire(q, "Syllogisme disjonctif", depuis=[1, 2])  # p∨q, ¬p ⊢ q
deriv.deduire(r, "Modus Ponens",          depuis=[4, 3])  # q, q→r ⊢ r
deriv.conclure()

n°,Formule,Justification
1.,$p \vee q$,Prémisse
2.,$\neg p$,Prémisse
3.,$q \Rightarrow r$,Prémisse
4.,$q$,"Syllogisme disjonctif (1, 2)"
5.,$r$,"Modus Ponens (4, 3)"


In [44]:
# L'étudiant fait une erreur de raisonnement
deriv = Deduction([p >> q, q], p, "Affirmation du conséquent (sophisme)")
deriv.deduire(p, "Modus Ponens (incorrect)", depuis=[1, 2])  # ❌ détecté
deriv.conclure()

n°,Formule,Justification
1.,$p \Rightarrow q$,Prémisse
2.,$q$,Prémisse
3.,$p$,"⚠️ Modus Ponens (incorrect) (1, 2) [INVALIDE]"
